# Modelos Baseline — Modelos Simples de Referencia**Objetivo:** Evaluar 3 modelos simples (Buy & Hold, Naive, Zero Prediction) para las 16 combinaciones de ventanas temporales. Estos resultados establecen el suelo mínimo de rendimiento que cualquier red neuronal debe superar para justificar su complejidad.**Métrica:** MAE (Mean Absolute Error)

In [ ]:
import sysimport os# Buscar el directorio raíz del proyecto localizando config.pydef find_project_root():    path = os.getcwd()    for _ in range(5):        if os.path.exists(os.path.join(path, 'config.py')):            return path        path = os.path.dirname(path)    return Noneproject_root = find_project_root()if project_root:    os.chdir(project_root)    sys.path.insert(0, project_root)    print(f"Directorio de trabajo: {os.getcwd()}")else:    print("ERROR: No se encontró el directorio raíz del proyecto")

## 1. Imports y configuración

In [ ]:
import numpy as npimport pandas as pd# Infraestructura compartida del proyectofrom src.data_pipeline import load_data, get_train_testfrom src.evaluation import compute_mae, load_all_resultsfrom src.baselines import (    predict_buy_and_hold,    predict_naive,    predict_zero,    evaluate_all_baselines)from config import INPUT_WINDOWS, OUTPUT_WINDOWS

## 2. Carga de datos

In [ ]:
returns = load_data()print(f"Shape: {returns.shape}")

## 3. Descripción de los modelos baselineSe implementan 3 modelos simples de referencia:- **Buy & Hold:** Predice la media histórica incondicional de los log-returns del conjunto de   entrenamiento. Asume que el mercado se comportará en el futuro como lo ha hecho en promedio   históricamente. Es el modelo simple requerido explícitamente por el enunciado.- **Naive (Last Value):** Predice que el promedio futuro de los log-returns será igual al último   valor observado en la ventana de entrada. Es el benchmark más básico en series temporales:   "mañana será como hoy".- **Zero Prediction:** Predice que el promedio futuro de los log-returns será cero. Los   log-returns diarios tienen media muy cercana a cero, lo que convierte a este modelo en un   baseline sorprendentemente competitivo, especialmente en horizontes cortos.Ninguno de estos modelos tiene parámetros entrenables. No hay curvas de convergencia porque no existe proceso de entrenamiento.

## 4. Evaluación de baselines para las 16 combinaciones de ventanasSe ejecutan los 3 baselines para cada combinación de ventanas de entrada (5, 10, 30, 90) y salida (1, 5, 30, 90), generando 48 resultados (3 modelos × 16 combinaciones). Todos se guardan automáticamente en un CSV exclusivo para baselines (`results/tables/baseline_results.csv`), separado de los resultados de redes neuronales para evitar sobreescrituras accidentales.

In [ ]:
evaluate_all_baselines(returns)

## 5. Tabla resumen de resultadosSe presentan los resultados de los 3 baselines organizados por combinación de ventanas. El MAE en test es la métrica que se usará para comparar contra los modelos de redes neuronales.

In [ ]:
# Cargar resultados de baselinesresults = load_all_results()# Filtrar solo baselinesbaselines_df = results[results['model_type'] == 'baseline'].copy()# Tabla pivotada: MAE test por modelo y combinación de ventanasbaselines_df['window_combo'] = baselines_df.apply(    lambda r: f"in={int(r['input_window'])}, out={int(r['output_window'])}", axis=1)pivot = baselines_df.pivot_table(    index='window_combo',    columns='model_name',    values='mae_test')# Ordenar las filas lógicamentewindow_order = [f"in={iw}, out={ow}" for iw in INPUT_WINDOWS for ow in OUTPUT_WINDOWS]pivot = pivot.reindex(window_order)print("MAE Test por modelo y combinación de ventanas:")print("=" * 65)print(pivot.to_string())

## 6. Mejor baseline por combinación de ventanasPara cada combinación de ventanas, se identifica qué modelo baseline obtiene el menor MAE en test. Este será el suelo que cada red neuronal deberá superar.

In [ ]:
# Identificar el mejor baseline por combinaciónbest_per_combo = baselines_df.loc[    baselines_df.groupby(['input_window', 'output_window'])['mae_test'].idxmin()][['input_window', 'output_window', 'model_name', 'mae_test']].reset_index(drop=True)best_per_combo.columns = ['Input Window', 'Output Window', 'Mejor Baseline', 'MAE Test']print("Mejor baseline por combinación:")print("=" * 65)print(best_per_combo.to_string(index=False))

## 7. AnálisisObservaciones clave a extraer de los resultados:1. **¿Qué baseline domina?** Identificar si Zero Prediction, Naive o Buy & Hold es    consistentemente mejor y en qué tipos de ventanas.2. **Efecto de la ventana de salida:** El MAE tiende a cambiar significativamente con la    ventana de salida (1d vs 90d), ya que promediar más días reduce la varianza de los    log-returns.3. **Efecto de la ventana de entrada:** Para Zero y Buy & Hold, la ventana de entrada tiene    impacto mínimo. Para Naive, sí influye porque la predicción depende del último valor    observado.4. **Referencia para redes neuronales:** Cualquier modelo de red neuronal que no bata al    mejor baseline para una combinación dada no está justificando su complejidad ni su coste    computacional.